In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from datetime import datetime, timezone
import json, os, pathlib, subprocess, sys, uuid
REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
BRANCH = 'Content-V9'
EXPECTED_EXACT = 'd9cd6932c3e9532453511203c5a2f5fcbefe8428'
RUNNER_MODULE = 'experiments.run_content_v9_stability'
ATTEMPT_NONCE = uuid.uuid4().hex[:12]
SOURCE = pathlib.Path(f'/content/cegwm-content-v9-source-{ATTEMPT_NONCE}')
LOCAL = pathlib.Path(f'/content/Content-V9-d9cd693-local-{ATTEMPT_NONCE}')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/CEG-WM/Content')
CAPTURE_LIMIT = 4096

def _v9_git(source, *args):
    return subprocess.run(['git', *args], cwd=source, check=True, capture_output=True, text=True).stdout.strip()

def detach_v9_execution_checkout(source, branch, expected_exact):
    handoff_head = _v9_git(source, 'rev-parse', 'HEAD')
    if _v9_git(source, 'branch', '--show-current') != branch or _v9_git(source, 'status', '--porcelain'):
        raise RuntimeError('canonical handoff identity')
    if subprocess.run(['git','merge-base','--is-ancestor',expected_exact,handoff_head], cwd=source, capture_output=True).returncode != 0:
        raise RuntimeError('execution exact is not handoff ancestor')
    _v9_git(source, 'checkout', '--detach', expected_exact)
    if _v9_git(source, 'rev-parse', 'HEAD') != expected_exact or _v9_git(source, 'branch', '--show-current') != '' or _v9_git(source, 'status', '--porcelain'):
        raise RuntimeError('checkout identity')
    return handoff_head

def allocate_v9_attempt_paths(content_root, drive_root, attempt_nonce, now_utc):
    source = content_root / f'cegwm-content-v9-source-{attempt_nonce}'
    local = content_root / f'Content-V9-d9cd693-local-{attempt_nonce}'
    if source.exists() or local.exists():
        raise FileExistsError('create-only local path')
    while True:
        run_utc = now_utc().strftime('%Y%m%dT%H%M%S%fZ')
        drive_target = drive_root / f'Content-V9-d9cd693-{run_utc}'
        if not drive_target.exists():
            return source, local, drive_target, run_utc


In [ ]:
SOURCE, LOCAL, DRIVE_TARGET, RUN_UTC = allocate_v9_attempt_paths(pathlib.Path('/content'), DRIVE_ROOT, ATTEMPT_NONCE, lambda: datetime.now(timezone.utc))
subprocess.run(['git','clone','--no-single-branch','--branch',BRANCH,REPO_URL,str(SOURCE)],check=True)
HANDOFF_HEAD = detach_v9_execution_checkout(SOURCE, BRANCH, EXPECTED_EXACT)
def git(*args): return _v9_git(SOURCE, *args)
subprocess.run([sys.executable,'-m','pip','install',str(SOURCE)],check=True)
if git('rev-parse','HEAD') != EXPECTED_EXACT or git('branch','--show-current') != '' or git('status','--porcelain') or LOCAL.exists() or DRIVE_TARGET.exists(): raise RuntimeError('post-install identity')
from google.colab import userdata
child_env={k:v for k,v in os.environ.items() if not any(x in k.upper() for x in ('TOKEN','KEY','SECRET','PASSWORD','CREDENTIAL'))}; root_key=token=''
try:
    root_key=userdata.get('CEG_WM_ROOT_KEY'); token=userdata.get('HF_TOKEN'); child_env['CEG_WM_ROOT_KEY']=root_key; child_env['HF_TOKEN']=token
    p=subprocess.Popen([sys.executable,'-m',RUNNER_MODULE,'--repo-root',str(SOURCE),'--expected-exact',EXPECTED_EXACT,'--local-work-root',str(LOCAL),'--artifact-sink',str(DRIVE_TARGET)],cwd=SOURCE,env=child_env,stdout=subprocess.PIPE,stderr=subprocess.DEVNULL)
finally:
    child_env.pop('CEG_WM_ROOT_KEY',None); child_env.pop('HF_TOKEN',None); root_key=token=''
summary=None; summary_count=0
for raw_line in iter(p.stdout.readline,b''):
    if len(raw_line)>CAPTURE_LIMIT: raise RuntimeError('runner line bound')
    line=raw_line.decode('utf-8','strict').strip()
    if line.startswith('CEGWM_SUMMARY '): summary=json.loads(line[14:]); summary_count+=1
rc=p.wait()
if summary_count!=1 or not isinstance(summary,dict) or summary.get('phase')!='terminal' or summary.get('rc')!=rc or not all(isinstance(summary.get(k),int) and not isinstance(summary.get(k),bool) for k in ('committed','fixed_total')): raise RuntimeError('terminal summary contract')


In [ ]:
if rc==0 and summary['committed']==summary['fixed_total']:
    print('CEGWM_CONTENT_V9_ARTIFACT '+json.dumps({'execution_exact':EXPECTED_EXACT,'artifact_result_path':str(DRIVE_TARGET),'completeness':'complete','scientific_status':'not_adjudicated'},sort_keys=True,separators=(',',':')))
elif rc==2:
    print('CEGWM_CONTENT_V9_INCOMPLETE '+json.dumps({'execution_exact':EXPECTED_EXACT,'artifact_result_path':str(DRIVE_TARGET),'completeness':'incomplete','scientific_status':'not_evaluable'},sort_keys=True,separators=(',',':')))
else: raise RuntimeError('runner completion contract')
